## SARIMA

In [ ]:
# ── Colab / Local setup ──────────────────────────────────────────────────
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 1. Clone the repo
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/DaTking4/ml-final-project-walmart-recruiting.git"], check=True)
    %cd ml-final-project-walmart-recruiting

    # 2. Install dependencies
    %pip install -q -r requirements.txt

    # 3. Inject Colab secrets as environment variables
    import os
    from google.colab import userdata
    os.environ["DAGSHUB_TOKEN"]    = userdata.get("DAGSHUB_TOKEN")
    os.environ["WANDB_API_KEY"]    = userdata.get("WANDB_API_KEY")

    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

    # 4. Download competition data
    %pip install -q kaggle
    import os
    os.makedirs("data", exist_ok=True)
    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data/ --quiet
    !unzip -q -o data/walmart-recruiting-store-sales-forecasting.zip -d data/

print("Running in:", "Google Colab" if IN_COLAB else "Local environment")


### 1. Setup and Imports

In [1]:
import os
import sys
import importlib
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))

repo_root = Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / "src").exists():
        sys.path.insert(0, str(repo_root))
        break
    repo_root = repo_root.parent
else:
    raise FileNotFoundError("Could not locate the repository root containing 'src'.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import mlflow
import mlflow.pyfunc
from mlflow.models import infer_signature

import wandb

import src.mlflow_setup as mlflow_setup
importlib.reload(mlflow_setup)
init_tracking = mlflow_setup.init_tracking

from src.data_loading import load_merged
from src.transforms import apply_shared_features
from src.validation import time_based_split
from src.pipeline.sarima_pipeline import SARIMAForecastPipeline
from src.arima_utils import (
    arima_order,
    evaluate_arima_config,
    fit_arima_models,
    forecast_one_series,
    to_arima_long,
)

init_tracking()
assert "dagshub.com" in mlflow.get_tracking_uri(), mlflow.get_tracking_uri()

print("Current MLflow URI:", mlflow.get_tracking_uri())

if mlflow.active_run() is not None:
    mlflow.end_run()

BLUE   = "#7196C7"
GREEN  = "#5E9D74"
RED    = "#7E3838"
PURPLE = "#705588"

print("Setup complete.")

c:\Users\l.chitishvili\Desktop\ml\ml-final-project-store-sales-forecasting\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
c:\Users\l.chitishvili\Desktop\ml\ml-final-project-store-sales-forecasting\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


Accessing as lchit22

Initialized MLflow to track repo "dkhak22/ml-final-project-store-sales-forecasting"

Repository dkhak22/ml-final-project-store-sales-forecasting initialized!

MLflow tracking URI: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow
Current MLflow URI: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow
Setup complete.


### 2. Configuration

In [2]:
init_tracking()
assert "dagshub.com" in mlflow.get_tracking_uri(), mlflow.get_tracking_uri()

EXPERIMENT_NAME = "SARIMA_Training"
mlflow.set_experiment(EXPERIMENT_NAME)

WANDB_ENTITY  = "dkhak22-free-university-of-tbilisi-"
WANDB_PROJECT = "walmart-sales-forecasting"

# s=52: annual seasonality for weekly retail data.
# SARIMA(p,d,q)(P,D,Q,52) adds seasonal AR/MA/differencing on top of ARIMA.
CONFIG = {
    "input_size":        52,
    "horizon":           26,
    "random_seed":       42,
    "min_train_points":  52,
    "maxiter":           80,
    "seasonal_period":   52,
}

FREQ      = "W-FRI"
MODEL_COL = "SARIMA"

CONFIG

Initialized MLflow to track repo "dkhak22/ml-final-project-store-sales-forecasting"

Repository dkhak22/ml-final-project-store-sales-forecasting initialized!

MLflow tracking URI: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow


{'input_size': 52,
 'horizon': 26,
 'random_seed': 42,
 'min_train_points': 52,
 'maxiter': 80,
 'seasonal_period': 52}

### 3. Load Data

In [3]:
train_df, test_df = load_merged()

print(f"train_df: {train_df.shape}")
print(f"test_df:  {test_df.shape}")

CONFIG["horizon"] = test_df["Date"].nunique()

train_df.head()

train_df: (421570, 16)
test_df:  (115064, 15)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


### 4. Shared Preprocessing and Feature Engineering

In [4]:
train_prepared = apply_shared_features(train_df)

print(f"train_prepared: {train_prepared.shape}")
train_prepared.head()

train_prepared: (421570, 23)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,Unemployment,Size,Type_A,Type_B,Type_C,Year,Month,WeekOfYear,DaysSinceLastHoliday,DaysToNextHoliday
0,1,1,2010-02-05,24924.50,0,42.31,2.572,0.0,0.0,0.0,...,8.106,151315,True,False,False,2010,2,5,inf,7.0
1,1,1,2010-02-12,46039.49,1,38.51,2.548,0.0,0.0,0.0,...,8.106,151315,True,False,False,2010,2,6,0.0,0.0
2,1,1,2010-02-19,41595.55,0,39.93,2.514,0.0,0.0,0.0,...,8.106,151315,True,False,False,2010,2,7,7.0,203.0
3,1,1,2010-02-26,19403.54,0,46.63,2.561,0.0,0.0,0.0,...,8.106,151315,True,False,False,2010,2,8,14.0,196.0
4,1,1,2010-03-05,21827.90,0,46.50,2.625,0.0,0.0,0.0,...,8.106,151315,True,False,False,2010,3,9,21.0,189.0


### 5. Feature Selection

In [5]:
# SARIMA is univariate — it models each Store-Dept series independently
# using only its own sales history. The seasonal component (P,D,Q,52)
# gives the model an explicit mechanism to learn annual patterns (holiday
# cycles, Christmas ramp-up, Super Bowl etc.) directly from the target
# series, without needing exogenous feature columns.
SARIMA_FEATURE_DECISION = {
    "feature_set": "target_history_only",
    "uses_exogenous_features": False,
    "used_model_columns": "unique_id, ds, y",
    "reason": (
        "SARIMA is a univariate statistical model. The seasonal component "
        "(P,D,Q,s=52) captures annual retail seasonality (Christmas, "
        "Thanksgiving, Super Bowl) directly from the sales history, making "
        "exogenous features unnecessary for this experiment."
    ),
}

SARIMA_FEATURE_DECISION

{'feature_set': 'target_history_only',
 'uses_exogenous_features': False,
 'used_model_columns': 'unique_id, ds, y',
 'reason': 'SARIMA is a univariate statistical model. The seasonal component (P,D,Q,s=52) captures annual retail seasonality (Christmas, Thanksgiving, Super Bowl) directly from the sales history, making exogenous features unnecessary for this experiment.'}

### 6. Time-Series and Window Setup

In [6]:
train_part, valid_part = time_based_split(
    train_prepared,
    valid_weeks=CONFIG["horizon"],
)

print(f"Train part: {train_part['Date'].min().date()} -> {train_part['Date'].max().date()}")
print(f"Valid part: {valid_part['Date'].min().date()} -> {valid_part['Date'].max().date()}")
print(f"\nInput size:       {CONFIG['input_size']} weeks")
print(f"Forecast horizon: {CONFIG['horizon']} weeks")
print(f"Seasonal period:  s = {CONFIG['seasonal_period']} (annual)")

long_df      = to_arima_long(train_prepared)
train_long   = to_arima_long(train_part)
valid_long   = to_arima_long(valid_part)

series_lengths = long_df.groupby("unique_id").size()
train_lengths  = train_long.groupby("unique_id").size()
valid_lengths  = valid_long.groupby("unique_id").size()

SARIMA_SERIES_LENGTH = int(series_lengths.max())
sarima_ids = series_lengths[series_lengths == SARIMA_SERIES_LENGTH].index
sarima_ids = sarima_ids.intersection(
    train_lengths[train_lengths >= CONFIG["min_train_points"]].index
)
sarima_ids = sarima_ids.intersection(
    valid_lengths[valid_lengths == CONFIG["horizon"]].index
)

sarima_df         = long_df[long_df["unique_id"].isin(sarima_ids)].copy()
sarima_train_long = train_long[train_long["unique_id"].isin(sarima_ids)].copy()
sarima_valid_long = valid_long[valid_long["unique_id"].isin(sarima_ids)].copy()

print("\nSeries length summary:")
display(series_lengths.describe())

print("\nSARIMA training/evaluation frame:")
print(f"Using {len(sarima_ids)} complete-history Store-Dept series")
print(f"Dropping {long_df['unique_id'].nunique() - len(sarima_ids)} short/ragged series")
print(f"Rows used by SARIMA: {sarima_df.shape[0]:,}")

holiday_lookup = train_prepared.assign(
    unique_id=train_prepared["Store"].astype(str) + "_" + train_prepared["Dept"].astype(str),
    ds=train_prepared["Date"],
)[["unique_id", "ds", "IsHoliday"]].drop_duplicates()
holiday_lookup["IsHoliday"] = holiday_lookup["IsHoliday"].fillna(False).astype(bool)

train_by_id = {
    uid: group.sort_values("ds").set_index("ds")["y"].astype(float).asfreq("W-FRI")
    for uid, group in sarima_train_long.groupby("unique_id")
}
valid_by_id = {
    uid: group.sort_values("ds")[["unique_id", "ds", "y"]].copy()
    for uid, group in sarima_valid_long.groupby("unique_id")
}

print("Prepared grouped SARIMA inputs.")


def sarima_order_str(config: dict) -> str:
    p, d, q = config["p"], config["d"], config["q"]
    so = config.get("seasonal_order", (0, 0, 0, 52))
    return f"({p},{d},{q})({so[0]},{so[1]},{so[2]},{so[3]})"


def fit_gap_pct(train_wmae, val_wmae):
    if pd.isna(train_wmae) or train_wmae == 0:
        return np.nan
    return ((val_wmae - train_wmae) / train_wmae) * 100


def classify_fit_status(train_wmae, val_wmae):
    gap = fit_gap_pct(train_wmae, val_wmae)
    if pd.isna(gap):
        return "unknown"
    if gap > 25:
        return "overfit"
    if gap < -10:
        return "underfit"
    return "good"

Train part: 2010-02-05 -> 2012-01-27
Valid part: 2012-02-03 -> 2012-10-26

Input size:       52 weeks
Forecast horizon: 39 weeks
Seasonal period:  s = 52 (annual)

Series length summary:


count    3331.000000
mean      126.559592
std        40.212763
min         1.000000
25%       143.000000
50%       143.000000
75%       143.000000
max       143.000000
dtype: float64


SARIMA training/evaluation frame:
Using 2660 complete-history Store-Dept series
Dropping 671 short/ragged series
Rows used by SARIMA: 380,380
Prepared grouped SARIMA inputs.


### 7. Forward and Backward Check

In [7]:
# Sanity check: SARIMA(1,1,0)(1,0,0,52) — simplest seasonal config
sanity_config = {
    "label":                 "sanity_sarima",
    "regime":                "sanity",
    "p": 1, "d": 1, "q": 0,
    "seasonal_order":        (1, 0, 0, 52),
    "trend":                 "n",
    "enforce_stationarity":  False,
    "enforce_invertibility": False,
    "concentrate_scale":     False,
    "maxiter":               20,
}

sample_id   = sarima_ids[0]
y_train     = train_by_id[sample_id]
valid_group = valid_by_id[sample_id]

forecast, failed = forecast_one_series(
    y_train=y_train,
    steps=len(valid_group),
    config=sanity_config,
    min_train_points=CONFIG["min_train_points"],
    fallback_value=float(y_train.iloc[-1]),
)

assert len(forecast) == CONFIG["horizon"]
assert np.isfinite(forecast).all()

print("SARIMA sanity check passed")
print("Used fallback:", failed)
pd.DataFrame({"ds": valid_group["ds"].to_numpy(), "forecast": forecast}).head()

SARIMA sanity check passed
Used fallback: False


,ds,forecast
0,2012-02-03,35666.307476
1,2012-02-10,42902.452171
2,2012-02-17,48267.456263
3,2012-02-24,31727.404159
4,2012-03-02,33110.275167


### 8. Baseline Run

In [9]:
# Baseline: SARIMA(1,1,1)(1,1,1,52) — the canonical seasonal ARIMA spec
baseline_config = {
    "label":                 "baseline_sarima_111_111_52",
    "regime":                "baseline",
    "p": 1, "d": 1, "q": 1,
    "seasonal_order":        (1, 1, 1, 52),
    "trend":                 "n",
    "enforce_stationarity":  False,
    "enforce_invertibility": False,
    "concentrate_scale":     False,
    "maxiter":               CONFIG["maxiter"],
}

with mlflow.start_run(run_name="SARIMA_Baseline") as run:
    wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name="SARIMA_Baseline",
        group="SARIMA",
        job_type="baseline",
        tags=["SARIMA", "baseline", "target_history_only"],
        config={**CONFIG, **baseline_config, **SARIMA_FEATURE_DECISION,
                "sarima_order": sarima_order_str(baseline_config)},
        reinit=True,
    )

    try:
        baseline_cv_df, baseline_val_wmae, baseline_failures, baseline_train_wmae = evaluate_arima_config(
            config=baseline_config,
            arima_ids=sarima_ids,
            train_by_id=train_by_id,
            valid_by_id=valid_by_id,
            holiday_lookup=holiday_lookup,
            model_col=MODEL_COL,
            min_train_points=CONFIG["min_train_points"],
        )

        gap_pct    = fit_gap_pct(baseline_train_wmae, baseline_val_wmae)
        fit_status = classify_fit_status(baseline_train_wmae, baseline_val_wmae)

        print(f"Baseline train WMAE:      {baseline_train_wmae:,.2f}")
        print(f"Baseline validation WMAE: {baseline_val_wmae:,.2f}")
        print(f"Baseline gap:             {gap_pct:,.2f}% ({fit_status})")
        print(f"Baseline fallback series: {baseline_failures:,}")

        mlflow.log_params({**CONFIG, **SARIMA_FEATURE_DECISION})
        mlflow.log_params({
            "label":                 baseline_config["label"],
            "regime":                baseline_config["regime"],
            "p":                     baseline_config["p"],
            "d":                     baseline_config["d"],
            "q":                     baseline_config["q"],
            "seasonal_order":        str(baseline_config["seasonal_order"]),
            "sarima_order":          sarima_order_str(baseline_config),
            "trend":                 baseline_config["trend"],
            "enforce_stationarity":  baseline_config["enforce_stationarity"],
            "enforce_invertibility": baseline_config["enforce_invertibility"],
            "concentrate_scale":     baseline_config["concentrate_scale"],
            "maxiter":               baseline_config["maxiter"],
            "sarima_n_series":       len(sarima_ids),
            "gradient_logging_applicable": False,
            "gradient_logging_reason": "SARIMA is a statistical model — no backpropagation.",
        })
        mlflow.log_metric("train_wmae",     baseline_train_wmae)
        mlflow.log_metric("val_wmae",       baseline_val_wmae)
        mlflow.log_metric("gap_pct",        gap_pct)
        mlflow.log_param("fit_status",      fit_status)
        mlflow.log_metric("fallback_series", baseline_failures)

        wandb.log({
            "train_wmae":     baseline_train_wmae,
            "val_wmae":       baseline_val_wmae,
            "gap_pct":        gap_pct,
            "fit_status":     fit_status,
            "fallback_series": baseline_failures,
        })

        # Fitted models are written one small file per series (see
        # fit_arima_models) instead of one combined joblib -- a single
        # ~22GB combined file for ~2,660 series reliably breaks the HTTP
        # PUT upload to DagsHub (SSLError/EOF mid-transfer). mlflow.log_artifacts
        # uploads each file in the directory individually instead.
        os.makedirs("artifacts", exist_ok=True)
        baseline_sarima_dir = Path("artifacts") / "sarima_baseline"
        baseline_models_dir = baseline_sarima_dir / "models"

        baseline_fitted_ids, baseline_fit_failures = fit_arima_models(
            full_long_df=sarima_train_long,
            ids=sarima_ids,
            config=baseline_config,
            models_dir=str(baseline_models_dir),
        )

        joblib.dump(
            {"config": baseline_config, "fitted_ids": baseline_fitted_ids,
             "failed_series": baseline_fit_failures},
            baseline_sarima_dir / "manifest.joblib",
        )

        mlflow.log_artifacts(str(baseline_sarima_dir), artifact_path="sarima_model")
        baseline_run_id = run.info.run_id

    finally:
        wandb.finish()

Evaluating 2,660 series for baseline_sarima_111_111_52 (n_jobs=-2)


[Parallel(n_jobs=-2)]: Using backend LokyBackend with 23 concurrent workers.
[Parallel(n_jobs=-2)]: Done   4 tasks      | elapsed:    5.8s
[Parallel(n_jobs=-2)]: Done 154 tasks      | elapsed:   28.2s


🏃 View run SARIMA_Baseline at: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow/#/experiments/4/runs/5f66e6da2c934d239436db90c662de66
🧪 View experiment at: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow/#/experiments/4


KeyboardInterrupt: 

### 9. Hyperparameters

In [8]:
# seasonal_order = (P, D, Q, s) where s=52 (annual, weekly data).
# D=1 applies seasonal differencing at lag 52 — removes annual trend.
# D=0 lets the model learn seasonality via P/Q terms without differencing.
# concentrate_scale=True significantly speeds up MLE optimization.

S = 52  # seasonal period

param_grid = [
    # --- Underfit: seasonal AR/MA without differencing, simple non-seasonal orders ---
    {"label": "underfit_1", "regime": "underfit", "p": 1, "d": 1, "q": 0, "seasonal_order": (1, 0, 0, S), "trend": "n", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 40},
    {"label": "underfit_2", "regime": "underfit", "p": 0, "d": 1, "q": 1, "seasonal_order": (0, 0, 1, S), "trend": "n", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 40},
    {"label": "underfit_3", "regime": "underfit", "p": 1, "d": 1, "q": 0, "seasonal_order": (0, 0, 1, S), "trend": "n", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 40},
   
    # --- Balanced: seasonal differencing (D=1) — the standard seasonal ARIMA approach ---
    {"label": "balanced_1",  "regime": "balanced", "p": 1, "d": 1, "q": 0, "seasonal_order": (1, 1, 0, S), "trend": "n", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 80},
    {"label": "balanced_2",  "regime": "balanced", "p": 0, "d": 1, "q": 1, "seasonal_order": (0, 1, 1, S), "trend": "n", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 80},
    {"label": "balanced_3",  "regime": "balanced", "p": 1, "d": 1, "q": 1, "seasonal_order": (1, 1, 0, S), "trend": "n", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 80},
    {"label": "balanced_4",  "regime": "balanced", "p": 1, "d": 1, "q": 1, "seasonal_order": (0, 1, 1, S), "trend": "n", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 80},
    {"label": "balanced_5",  "regime": "balanced", "p": 1, "d": 1, "q": 1, "seasonal_order": (1, 1, 1, S), "trend": "n", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 80},
    {"label": "balanced_6",  "regime": "balanced", "p": 2, "d": 1, "q": 0, "seasonal_order": (1, 1, 0, S), "trend": "n", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 80},
    {"label": "balanced_7",  "regime": "balanced", "p": 0, "d": 1, "q": 2, "seasonal_order": (0, 1, 1, S), "trend": "n", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 80},
    {"label": "balanced_8",  "regime": "balanced", "p": 2, "d": 1, "q": 1, "seasonal_order": (1, 1, 1, S), "trend": "n", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 80},
    {"label": "balanced_9", "regime": "balanced", "p": 2, "d": 0, "q": 1, "seasonal_order": (1, 1, 0, S), "trend": "c", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": True,  "maxiter": 120},
    {"label": "balanced_10", "regime": "balanced", "p": 1, "d": 0, "q": 2, "seasonal_order": (0, 1, 1, S), "trend": "c", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": True,  "maxiter": 120},

    # --- Complex: higher non-seasonal orders with full seasonal component ---
    {"label": "complex_1",   "regime": "complex", "p": 3, "d": 1, "q": 1, "seasonal_order": (1, 1, 1, S), "trend": "c", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 120},
    {"label": "complex_2",   "regime": "complex", "p": 1, "d": 1, "q": 3, "seasonal_order": (1, 1, 1, S), "trend": "c", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 120},
    {"label": "complex_3",   "regime": "complex", "p": 2, "d": 1, "q": 2, "seasonal_order": (1, 1, 1, S), "trend": "c", "enforce_stationarity": False, "enforce_invertibility": False, "concentrate_scale": False, "maxiter": 160},
]

print(f"Total SARIMA configs: {len(param_grid)}")

Total SARIMA configs: 16


### 10. SARIMA Experiments

In [9]:
results     = []
cv_by_label = {}

best_val_wmae        = float("inf")
best_run_id          = None
best_label           = None

with mlflow.start_run(run_name="SARIMA_HyperparamSweep") as parent_run:
    mlflow.log_param("n_configs",      len(param_grid))
    mlflow.log_param("model",          "SARIMA")
    mlflow.log_param("feature_set",    SARIMA_FEATURE_DECISION["feature_set"])
    mlflow.log_param("sarima_n_series", len(sarima_ids))
    mlflow.log_param("seasonal_period", CONFIG["seasonal_period"])
    mlflow.log_param("gradient_logging_applicable", False)

    for config in param_grid:
        label  = config["label"]
        regime = config["regime"]
        order  = sarima_order_str(config)

        with mlflow.start_run(run_name=f"SARIMA_{label}", nested=True) as nested_run:
            wandb.init(
                entity=WANDB_ENTITY,
                project=WANDB_PROJECT,
                name=f"SARIMA_{label}",
                group="SARIMA",
                job_type="hyperparameter_sweep",
                tags=["SARIMA", regime, "target_history_only"],
                config={**CONFIG, **config, "sarima_order": order, **SARIMA_FEATURE_DECISION},
                reinit=True,
            )

            try:
                cv_df, val_wmae, failures, train_wmae = evaluate_arima_config(
                    config=config,
                    arima_ids=sarima_ids,
                    train_by_id=train_by_id,
                    valid_by_id=valid_by_id,
                    holiday_lookup=holiday_lookup,
                    model_col=MODEL_COL,
                    min_train_points=CONFIG["min_train_points"],
                )
                cv_by_label[label] = cv_df.copy()

                gap_pct    = fit_gap_pct(train_wmae, val_wmae)
                fit_status = classify_fit_status(train_wmae, val_wmae)

                mlflow.log_params({
                    "label":                 label,
                    "regime":                regime,
                    "p":                     config["p"],
                    "d":                     config["d"],
                    "q":                     config["q"],
                    "seasonal_order":        str(config["seasonal_order"]),
                    "sarima_order":          order,
                    "trend":                 config["trend"],
                    "enforce_stationarity":  config["enforce_stationarity"],
                    "enforce_invertibility": config["enforce_invertibility"],
                    "concentrate_scale":     config["concentrate_scale"],
                    "maxiter":               config["maxiter"],
                    "feature_set":           SARIMA_FEATURE_DECISION["feature_set"],
                    "uses_exogenous_features": SARIMA_FEATURE_DECISION["uses_exogenous_features"],
                    "sarima_n_series":       len(sarima_ids),
                    "gradient_logging_applicable": False,
                })
                mlflow.log_metric("train_wmae",      train_wmae)
                mlflow.log_metric("val_wmae",        val_wmae)
                mlflow.log_metric("gap_pct",         gap_pct)
                mlflow.log_param("fit_status",       fit_status)
                mlflow.log_metric("fallback_series", failures)

                wandb.log({
                    "train_wmae":     train_wmae,
                    "val_wmae":       val_wmae,
                    "gap_pct":        gap_pct,
                    "fit_status":     fit_status,
                    "fallback_series": failures,
                })

                results.append({
                    "label":                 label,
                    "regime":                regime,
                    "p":                     config["p"],
                    "d":                     config["d"],
                    "q":                     config["q"],
                    "seasonal_order":        str(config["seasonal_order"]),
                    "sarima_order":          order,
                    "trend":                 config["trend"],
                    "enforce_stationarity":  config["enforce_stationarity"],
                    "enforce_invertibility": config["enforce_invertibility"],
                    "concentrate_scale":     config["concentrate_scale"],
                    "maxiter":               config["maxiter"],
                    "feature_set":           SARIMA_FEATURE_DECISION["feature_set"],
                    "train_wmae":            train_wmae,
                    "val_wmae":              val_wmae,
                    "gap_pct":               gap_pct,
                    "fit_status":            fit_status,
                    "fallback_series":       failures,
                    "run_id":               nested_run.info.run_id,
                })

                if val_wmae < best_val_wmae:
                    best_val_wmae  = val_wmae
                    best_run_id   = nested_run.info.run_id
                    best_label    = label

                    # One small file per series (see fit_arima_models) instead
                    # of one combined joblib -- a single ~22GB combined file
                    # for ~2,660 series reliably breaks the HTTP PUT upload to
                    # DagsHub. mlflow.log_artifacts uploads each file in the
                    # directory individually instead.
                    os.makedirs("artifacts", exist_ok=True)
                    checkpoint_dir = Path("artifacts") / f"sarima_{label}"
                    checkpoint_models_dir = checkpoint_dir / "models"

                    checkpoint_fitted_ids, checkpoint_failures = fit_arima_models(
                        full_long_df=sarima_train_long,
                        ids=sarima_ids,
                        config=config,
                        models_dir=str(checkpoint_models_dir),
                    )
                    joblib.dump(
                        {"config": config, "fitted_ids": checkpoint_fitted_ids,
                         "failed_series": checkpoint_failures},
                        checkpoint_dir / "manifest.joblib",
                    )
                    mlflow.log_artifacts(str(checkpoint_dir), artifact_path="sarima_model")

                    print(
                        f"New best: {best_label} | "
                        f"order = {order} | "
                        f"val WMAE = {best_val_wmae:,.2f}"
                    )

            finally:
                wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\l.chitishvili\_netrc.
wandb: Currently logged in as: lchit22 (lchit22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Evaluating 2,660 series for underfit_1 (n_jobs=-2)


[Parallel(n_jobs=-2)]: Using backend LokyBackend with 23 concurrent workers.
[Parallel(n_jobs=-2)]: Done   4 tasks      | elapsed:    2.8s
[Parallel(n_jobs=-2)]: Done 154 tasks      | elapsed:    7.1s
[Parallel(n_jobs=-2)]: Done 404 tasks      | elapsed:   14.1s
[Parallel(n_jobs=-2)]: Done 754 tasks      | elapsed:   23.9s
[Parallel(n_jobs=-2)]: Done 1204 tasks      | elapsed:   35.9s
[Parallel(n_jobs=-2)]: Done 1754 tasks      | elapsed:   49.8s
[Parallel(n_jobs=-2)]: Done 2404 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-2)]: Done 2660 out of 2660 | elapsed:  1.2min finished


Fitting 2,660 final ARIMA models (n_jobs=-2) -> artifacts\sarima_underfit_1\models


[Parallel(n_jobs=-2)]: Using backend LokyBackend with 23 concurrent workers.
[Parallel(n_jobs=-2)]: Done   4 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-2)]: Done 154 tasks      | elapsed:    4.3s
[Parallel(n_jobs=-2)]: Done 404 tasks      | elapsed:   10.8s
[Parallel(n_jobs=-2)]: Done 754 tasks      | elapsed:   20.1s
[Parallel(n_jobs=-2)]: Done 1204 tasks      | elapsed:   31.7s
[Parallel(n_jobs=-2)]: Done 1754 tasks      | elapsed:   45.6s
[Parallel(n_jobs=-2)]: Done 2404 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-2)]: Done 2660 out of 2660 | elapsed:  1.1min finished


...

Fitting 2,660 final ARIMA models (n_jobs=-2) -> artifacts\sarima_balanced_1\models


[Parallel(n_jobs=-2)]: Using backend LokyBackend with 23 concurrent workers.
[Parallel(n_jobs=-2)]: Done   4 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-2)]: Done 154 tasks      | elapsed:    8.3s
[Parallel(n_jobs=-2)]: Done 404 tasks      | elapsed:   21.3s
[Parallel(n_jobs=-2)]: Done 754 tasks      | elapsed:   40.0s
[Parallel(n_jobs=-2)]: Done 1204 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-2)]: Done 1754 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-2)]: Done 2404 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-2)]: Done 2660 out of 2660 | elapsed:  2.3min finished


fallback_series,▁
gap_pct,▁
train_wmae,▁
val_wmae,▁
fallback_series,0
fit_status,underfit
gap_pct,-32.05301
train_wmae,2897.36165
val_wmae,1968.67015


🏃 View run SARIMA_balanced_1 at: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow/#/experiments/4/runs/03375f1b61884555b8dc8b79daa9ceda
🧪 View experiment at: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow/#/experiments/4
🏃 View run SARIMA_HyperparamSweep at: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow/#/experiments/4/runs/cfa953eeaaba4b27b62c842e6075b1aa
🧪 View experiment at: https://dagshub.com/dkhak22/ml-final-project-store-sales-forecasting.mlflow/#/experiments/4


KeyboardInterrupt: 

### 11. Results

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("val_wmae").reset_index(drop=True)

display_cols = [
    "label", "regime", "sarima_order", "trend",
    "enforce_stationarity", "enforce_invertibility", "concentrate_scale",
    "maxiter", "feature_set", "val_wmae", "fallback_series",
]

display(results_df[display_cols])

os.makedirs("reports", exist_ok=True)
results_path = "reports/sarima_results.csv"
results_df.to_csv(results_path, index=False)

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(results_path)

### 12. Plots

In [ ]:
os.makedirs("Plots", exist_ok=True)

top_runs = results_df.sort_values("val_wmae").copy()
colors   = top_runs["label"].map(lambda l: GREEN if l == best_label else BLUE)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_runs["sarima_order"] + " " + top_runs["label"], top_runs["val_wmae"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("Validation WMAE")
ax.set_title("SARIMA - Validation WMAE by Configuration")
ax.grid(axis="x", alpha=0.3)

for idx, value in enumerate(top_runs["val_wmae"]):
    if np.isfinite(value) and value < 1e10:
        ax.text(value + 25, idx, f"{value:,.0f}", va="center", fontsize=8)

plt.tight_layout()
plot_path = "Plots/sarima_wmae_comparison.png"
plt.savefig(plot_path, dpi=200)
plt.show()

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(plot_path)

wandb.init(
    entity=WANDB_ENTITY, project=WANDB_PROJECT,
    name="SARIMA_Analysis", group="SARIMA", job_type="analysis",
    tags=["SARIMA", "analysis", "validation_wmae"], reinit=True,
)
wandb.log({"sarima_wmae_comparison": wandb.Image(plot_path), "best_val_wmae": best_val_wmae})
wandb.finish()

### 13. Error Analysis

In [ ]:
best_cv_df = cv_by_label[best_label].copy()
best_cv_df["abs_error"] = (best_cv_df["y"] - best_cv_df[MODEL_COL]).abs()
best_cv_df[["Store", "Dept"]] = best_cv_df["unique_id"].str.split("_", n=1, expand=True)

display(best_cv_df.head())

worst_store_dept = (
    best_cv_df.groupby(["Store", "Dept"])["abs_error"]
    .mean().sort_values(ascending=False).head(15)
)
display(worst_store_dept)

holiday_error = best_cv_df.groupby("IsHoliday")["abs_error"].mean()
display(holiday_error)

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_metric("holiday_week_mae",    float(holiday_error.get(True, np.nan)))
    mlflow.log_metric("nonholiday_week_mae", float(holiday_error.get(False, np.nan)))

### 14. Error Plots

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
worst_store_dept.sort_values().plot(kind="barh", ax=ax, color=RED)
ax.set_xlabel("Mean Absolute Error")
ax.set_title("SARIMA — Worst Store/Dept Validation Errors")
ax.grid(True, alpha=0.3)
plt.tight_layout()

error_plot_path = "Plots/sarima_worst_store_dept.png"
plt.savefig(error_plot_path, dpi=200)
plt.show()

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(error_plot_path)

wandb.init(
    entity=WANDB_ENTITY, project=WANDB_PROJECT,
    name="SARIMA_Error_Analysis", group="SARIMA", job_type="analysis",
    tags=["SARIMA", "error_analysis"], reinit=True,
)
wandb.log({
    "sarima_worst_store_dept": wandb.Image(error_plot_path),
    "holiday_week_mae":    float(holiday_error.get(True, np.nan)),
    "nonholiday_week_mae": float(holiday_error.get(False, np.nan)),
})
wandb.finish()

### 15. Best Model

In [ ]:
print("Best label:",           best_label)
print("Best run id:",          best_run_id)
print("Best validation WMAE:", best_val_wmae)

assert best_label  is not None
assert best_run_id is not None

best_row    = results_df[results_df["label"] == best_label].iloc[0]
best_config = next(c for c in param_grid if c["label"] == best_label)

print("Best config:")
display(best_config)

fallback_by_id  = (
    to_arima_long(train_prepared)
    .sort_values("ds")
    .groupby("unique_id")["y"]
    .last().astype(float).to_dict()
)
global_fallback = float(to_arima_long(train_prepared)["y"].median())

print(f"Fallback values available for {len(fallback_by_id)} series")
print(f"Global fallback Weekly_Sales: {global_fallback:,.2f}")

# One small file per series (see fit_arima_models) instead of one combined
# joblib -- a single ~22GB combined file for ~2,660 series reliably breaks
# the HTTP PUT upload to DagsHub (SSLError/EOF mid-transfer), and would also
# require holding every fitted model in memory simultaneously just to dump
# them together. mlflow.log_artifacts/log_model upload each file in the
# directory individually instead, and SARIMAForecastPipeline lazily loads
# one series' model at a time during predict().
best_model_dir   = Path("artifacts") / f"sarima_final_{best_label}"
final_models_dir = best_model_dir / "models"

final_fitted_ids, final_failures = fit_arima_models(
    full_long_df=sarima_df,
    ids=sarima_ids,
    config=best_config,
    models_dir=str(final_models_dir),
)

print(f"Final SARIMA models fitted: {len(final_fitted_ids):,}")
print(f"Final SARIMA failed series: {len(final_failures):,}")

joblib.dump(
    {"best_config": best_config, "fitted_ids": final_fitted_ids,
     "fallback_by_id": fallback_by_id, "global_fallback": global_fallback,
     "failed_series": final_failures},
    best_model_dir / "manifest.joblib",
)
print("Final SARIMA model artifacts saved to:", best_model_dir)

pipeline_model = SARIMAForecastPipeline(
    fallback_by_id=fallback_by_id,
    global_fallback=global_fallback,
)

class _SignatureContext:
    artifacts = {"sarima_model_dir": str(best_model_dir)}

_temp = SARIMAForecastPipeline(
    fallback_by_id=fallback_by_id,
    global_fallback=global_fallback,
)
_temp.load_context(_SignatureContext())
sample_output = _temp.predict(_SignatureContext(), test_df)
signature     = infer_signature(test_df, sample_output)

display(sample_output.head())

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_param("registered_model_name",   "SARIMA_WalmartForecast")
    mlflow.log_metric("best_val_wmae",           best_val_wmae)
    mlflow.log_metric("final_failed_series",     len(final_failures))
    mlflow.log_param("best_label",               best_label)
    mlflow.log_param("best_sarima_order",        sarima_order_str(best_config))
    mlflow.log_param("best_seasonal_order",      str(best_config["seasonal_order"]))
    mlflow.log_param("best_p",                   best_config["p"])
    mlflow.log_param("best_d",                   best_config["d"])
    mlflow.log_param("best_q",                   best_config["q"])
    mlflow.log_param("best_trend",               best_config["trend"])
    mlflow.log_param("best_concentrate_scale",   best_config["concentrate_scale"])
    mlflow.log_param("best_maxiter",             best_config["maxiter"])
    mlflow.log_param("final_sarima_n_series",    len(sarima_ids))
    mlflow.log_param("fallback_method",          "last_observed_sales_by_store_dept")
    mlflow.log_metric("global_fallback_weekly_sales", global_fallback)
    mlflow.log_param("gradient_logging_applicable", False)

    logged_model_info = mlflow.pyfunc.log_model(
        artifact_path="pipeline",
        python_model=pipeline_model,
        artifacts={"sarima_model_dir": str(best_model_dir)},
        code_paths=[str(repo_root / "src")],
        signature=signature,
        input_example=test_df.head(20),
        registered_model_name="SARIMA_WalmartForecast",
    )

model_uri = logged_model_info.model_uri
print("Logged model URI:", model_uri)

### 16. Test Loading

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_uri)
loaded_preds = loaded_model.predict(test_df.head(200))

display(loaded_preds.head())
print("Loaded prediction shape:", loaded_preds.shape)
assert set(loaded_preds.columns) == {"Id", "Weekly_Sales"}
assert loaded_preds["Weekly_Sales"].notna().all()